<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/BIGLUST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BigLust (SDXL)

1. **Runtime → Change runtime type → T4 GPU.**
2. Run **cell 1** (install + downloads; allow Google Drive access), **cell 2** (saves the server program) and
   **cell 3** (starts it — prints the `READY:` URL after BigLust loads).
3. Paste the URL into the website's **Backend URL** field. Add more SDXL checkpoints and LoRAs from the website,
   use the presets, and turn on **Face fix** for sharper faces that look more like a person LoRA.

The server runs as a separate program: if a generation runs out of memory, only the server restarts (same URL,
the model reloads in about a minute) and this notebook keeps running. **Cell 4** shows the server's log and memory use.

In [ ]:
# Cell 1: install ComfyUI and download BigLust
%cd /content
# --- Google Drive: keep downloaded LoRAs between Colab sessions (set to False to skip) ---
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")  # Colab asks you to allow access

!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
!git clone https://github.com/city96/ComfyUI-GGUF /content/ComfyUI/custom_nodes/ComfyUI-GGUF
!pip install -q torchsde gguf
!pip install -q -r /content/ComfyUI/requirements.txt
!apt install aria2 -qqy

!mkdir -p /content/ComfyUI/models/checkpoints /content/ComfyUI/models/loras

# Store LoRAs in MyDrive/ComfyUI/loras_sdxl: ComfyUI's loras folder becomes a link to it, so LoRAs
# downloaded from the website are saved to Drive and show up again in every new session.
import os, shutil
LORA_LOCAL = "/content/ComfyUI/models/loras"
if USE_GOOGLE_DRIVE:
    DRIVE_LORAS = "/content/drive/MyDrive/ComfyUI/loras_sdxl"
    os.makedirs(DRIVE_LORAS, exist_ok=True)
    if not os.path.islink(LORA_LOCAL):
        if os.path.isdir(LORA_LOCAL):
            for f in os.listdir(LORA_LOCAL):  # keep anything already downloaded in this session
                src, dst = os.path.join(LORA_LOCAL, f), os.path.join(DRIVE_LORAS, f)
                if not f.startswith("put_") and not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(LORA_LOCAL)
        os.symlink(DRIVE_LORAS, LORA_LOCAL)
    n = len([f for f in os.listdir(DRIVE_LORAS) if not f.endswith(".part")])
    print(f"LoRAs are stored in Google Drive (MyDrive/ComfyUI/loras_sdxl): {n} file(s)")
else:
    os.makedirs(LORA_LOCAL, exist_ok=True)
    print("LoRAs are stored on this runtime only (lost when the runtime is deleted)")

# Download BigLust (SDXL checkpoint)
!wget -q --show-progress -O /content/ComfyUI/models/checkpoints/biglust.safetensors \
  "https://civitai.com/api/download/models/1081768?fileId=986850&token=777a5632707c15ecf54e13eb3b1b862b"

# Download VAE (SDXL VAE is recommended for BigLust)
!mkdir -p /content/ComfyUI/models/vae
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors \
  -d /content/ComfyUI/models/vae -o sdxl_vae.safetensors


# Face fix (website toggle): the face detector ADetailer uses
!pip install -q ultralytics
!mkdir -p /content/ComfyUI/models/ultralytics
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt \
  -d /content/ComfyUI/models/ultralytics -o face_yolov8m.pt

# Check downloads
!ls -lh /content/ComfyUI/models/checkpoints/
!ls -lh /content/ComfyUI/models/loras/
!ls -lh /content/ComfyUI/models/vae/

In [ ]:
%%writefile /content/biglust_server.py
# Cell 2: the generation server, saved as its own program (/content/biglust_server.py).
# Cell 3 runs it separately from this notebook: if a generation runs out of memory only this program
# dies, and cell 3 restarts it automatically (same URL) instead of the whole Colab session crashing.
# Running this cell only saves the file; edit it here and re-run cells 2 and 3 to apply changes.
# Add any SDXL (or SD1.5) checkpoint or LoRA from the website with "+ Download Checkpoint" / "+ Download LoRA".
import os
import sys

os.chdir("/content/ComfyUI")
sys.path.insert(0, "/content/ComfyUI")

CRASH_FILE = "/content/server_crash.txt"


def last_crash_message():
    """Set by cell 3 when the server crashed and was restarted, so the website can say why a job vanished."""
    try:
        return open(CRASH_FILE).read().strip()
    except OSError:
        return ""


print("Loading BigLust...", flush=True)
import torch
import random, time
from PIL import Image
import numpy as np
import logging
import warnings

# Hide warnings
logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

from nodes import NODE_CLASS_MAPPINGS

CheckpointLoaderSimple = NODE_CLASS_MAPPINGS["CheckpointLoaderSimple"]()
CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()

with torch.inference_mode():
    # Load BigLust checkpoint
    ckpt = CheckpointLoaderSimple.load_checkpoint("biglust.safetensors")
    unet = ckpt[0]
    clip = ckpt[1]
    vae = ckpt[2]

print("BigLust loaded.", flush=True)


# ---------------------------------------------------------------- API for the website
import os
import re
import gc
import time
import uuid
import base64
import random
import threading
import subprocess
import traceback
import urllib.parse
from io import BytesIO
from typing import List, Optional

import numpy as np
import requests
import torch
from PIL import Image
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

import comfy.utils
import comfy.model_management
from nodes import NODE_CLASS_MAPPINGS

# Optional: your Civitai API key (some models need it). The website also sends one.
CIVITAI_TOKEN = os.environ.get("CIVITAI_TOKEN", "")

MODELS_DIR = "/content/ComfyUI/models"
CKPT_DIR = os.path.join(MODELS_DIR, "checkpoints")
LORA_DIR = os.path.join(MODELS_DIR, "loras")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LORA_DIR, exist_ok=True)
MODEL_EXTS = (".safetensors", ".ckpt", ".pt", ".pth", ".bin")
DEFAULT_CHECKPOINT = "biglust.safetensors"

CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
CheckpointLoaderSimple = NODE_CLASS_MAPPINGS["CheckpointLoaderSimple"]()

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ---------------- checkpoints: keep exactly one loaded ----------------
loaded = globals().get("loaded") if isinstance(globals().get("loaded"), dict) else {"name": None, "model": None, "clip": None, "vae": None}
if loaded["name"] is None and "unet" in globals() and "clip" in globals() and "vae" in globals():
    loaded.update(name=DEFAULT_CHECKPOINT, model=unet, clip=clip, vae=vae)  # loaded by cell 3


def list_files(folder):
    out = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(MODEL_EXTS):
                out.append(os.path.relpath(os.path.join(root, f), folder))
    return sorted(out, key=str.lower)


def get_checkpoint(name):
    name = name or DEFAULT_CHECKPOINT
    if loaded["name"] == name:
        return loaded["model"], loaded["clip"], loaded["vae"]
    if name not in list_files(CKPT_DIR):
        raise ValueError(f"Checkpoint not found: {name}")
    print(f"Switching checkpoint: {loaded['name']} -> {name}")
    # free the previous checkpoint first so two SDXL models never sit in RAM together
    loaded.update(name=None, model=None, clip=None, vae=None)
    for g in ("unet", "clip", "vae"):
        globals().pop(g, None)
    comfy.model_management.unload_all_models()
    gc.collect()
    comfy.model_management.soft_empty_cache()
    with torch.inference_mode():
        m, c, v = CheckpointLoaderSimple.load_checkpoint(name)[:3]
    loaded.update(name=name, model=m, clip=c, vae=v)
    return m, c, v


# ---------------- downloads (background, streamed to disk) ----------------
JOBS = globals().get("JOBS") if isinstance(globals().get("JOBS"), dict) else {}  # kept across re-runs


def new_job(kind, **extra):
    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {"kind": kind, "status": "queued", "created": time.time(), "error": None, **extra}
    now = time.time()
    for jid in [j for j, v in JOBS.items() if v["status"] in ("done", "error") and now - v["created"] > 1800]:
        JOBS.pop(jid, None)
    return job_id


def resolve_url(url):
    """Turn page links into direct download links. Returns (download_url, suggested_name)."""
    url = url.strip()
    parsed = urllib.parse.urlparse(url)
    qs = urllib.parse.parse_qs(parsed.query)
    token = (qs.get("token") or [CIVITAI_TOKEN])[0]
    host = parsed.netloc.lower()

    if "civitai.com" in host or "civitai.red" in host:
        api_headers = {"Authorization": f"Bearer {token}"} if token else {}
        m = re.search(r"/api/download/models/(\d+)", parsed.path)
        if not m:
            version_id = (qs.get("modelVersionId") or [None])[0]
            if not version_id:
                mm = re.search(r"/models/(\d+)", parsed.path)
                if not mm:
                    raise ValueError("Unrecognised Civitai link. Use a model page or download link.")
                info = requests.get(f"https://civitai.com/api/v1/models/{mm.group(1)}", headers=api_headers, timeout=30)
                info.raise_for_status()
                versions = info.json().get("modelVersions") or []
                if not versions:
                    raise ValueError("That Civitai model has no downloadable versions.")
                version_id = versions[0]["id"]  # newest version
            url = f"https://civitai.com/api/download/models/{version_id}"
        else:
            url = f"https://civitai.com/api/download/models/{m.group(1)}"
        if token:
            url += "?token=" + urllib.parse.quote(token)
        return url, None

    if "huggingface.co" in host:
        url = url.replace("/blob/", "/resolve/")
        return url, os.path.basename(urllib.parse.urlparse(url).path)

    return url, os.path.basename(parsed.path)


def filename_from_response(resp, fallback):
    cd = resp.headers.get("content-disposition", "")
    m = re.search(r"filename\*=(?:UTF-8'')?([^;]+)", cd, re.I) or re.search(r'filename="?([^";]+)"?', cd, re.I)
    name = urllib.parse.unquote(m.group(1).strip()) if m else (fallback or "")
    name = os.path.basename(name).strip().strip('"')
    name = re.sub(r"[^\w.\-+() ]", "_", name)
    return name


def run_download(job_id, url, kind):
    job = JOBS[job_id]
    job["status"] = "running"
    folder = LORA_DIR if kind == "lora" else CKPT_DIR
    tmp_path = None
    try:
        dl_url, suggested = resolve_url(url)
        with requests.get(dl_url, stream=True, timeout=60, allow_redirects=True,
                          headers={"User-Agent": "Mozilla/5.0"}) as r:
            if r.status_code in (401, 403):
                raise ValueError("Access denied — this model needs a Civitai API key (set CIVITAI_TOKEN or add ?token=...)")
            r.raise_for_status()
            if "text/html" in r.headers.get("content-type", ""):
                raise ValueError("The link returned a web page, not a model file. Use the download link.")
            name = filename_from_response(r, suggested)
            if not name.lower().endswith(MODEL_EXTS):
                name = f"{kind}_{int(time.time())}.safetensors"
            job["filename"] = name
            path = os.path.join(folder, name)
            if os.path.exists(path):
                job["status"], job["progress"] = "done", 100
                return
            total = int(r.headers.get("content-length") or 0)
            job["total_mb"] = round(total / 1e6) if total else None
            tmp_path = path + ".part"
            done = 0
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    f.write(chunk)
                    done += len(chunk)
                    job["done_mb"] = round(done / 1e6)
                    job["progress"] = round(done * 100 / total) if total else None
            os.replace(tmp_path, path)
            tmp_path = None
        job["status"], job["progress"] = "done", 100
        print(f"Downloaded {kind}: {name} ({job.get('done_mb')} MB)")
    except Exception as e:
        traceback.print_exc()
        job["status"], job["error"] = "error", str(e)
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)


class ModelDownload(BaseModel):
    url: str
    type: str = "checkpoint"


@app.post("/download_model")
def download_model(req: ModelDownload):
    kind = "lora" if req.type == "lora" else "checkpoint"
    job_id = new_job("download", type=kind, filename=None, progress=0, done_mb=0, total_mb=None)
    threading.Thread(target=run_download, args=(job_id, req.url, kind), daemon=True).start()
    return {"job_id": job_id, "status": "queued", "type": kind}


@app.get("/user_models")
def get_user_models():
    ckpts = list_files(CKPT_DIR)
    if DEFAULT_CHECKPOINT in ckpts:  # default first
        ckpts.remove(DEFAULT_CHECKPOINT)
        ckpts.insert(0, DEFAULT_CHECKPOINT)
    return {
        "checkpoints": ckpts,
        "lora_files": list_files(LORA_DIR),
        "model_type": "sdxl",
        "supports_face_fix": face_fix_available(),
        "default_checkpoint": DEFAULT_CHECKPOINT,
        "samplers": ["dpmpp_2m", "dpmpp_2m_sde", "dpmpp_sde", "euler", "euler_ancestral", "dpmpp_3m_sde", "uni_pc"],
        "schedulers": ["karras", "normal", "simple", "exponential", "sgm_uniform"],
    }


# ---------------- LoRA loading with a compatibility check ----------------
# ComfyUI silently skips LoRA layers that don't exist in the model, so a LoRA made for another
# model (e.g. SDXL on Qwen) would quietly do nothing. Count what matches and say so.
import comfy.lora
import comfy.lora_convert
import comfy.sd
import folder_paths

_LORA_SUFFIXES = (".lora_up.weight", ".lora_down.weight", ".lora_A.weight", ".lora_B.weight",
                  ".lora_mid.weight", ".lora.up.weight", ".lora.down.weight", ".diff", ".hada_w1_a",
                  ".lokr_w1", ".lokr_w1_a", ".oft_blocks", ".w_norm", ".set_weight")


def apply_lora_checked(model, clip, name, strength_model, strength_clip, model_label):
    """Apply a LoRA like ComfyUI's LoraLoader. Returns (model, clip, warning_or_None).
    Raises ValueError if none of the LoRA's layers match the model."""
    path = folder_paths.get_full_path_or_raise("loras", name)
    lora, metadata = comfy.utils.load_torch_file(path, safe_load=True, return_metadata=True)
    key_map = {}
    if model is not None:
        key_map = comfy.lora.model_lora_keys_unet(model.model, key_map)
    if clip is not None:
        key_map = comfy.lora.model_lora_keys_clip(clip.cond_stage_model, key_map)
    converted = comfy.lora_convert.convert_lora(dict(lora))
    patches = comfy.lora.load_lora(converted, key_map, log_missing=False)
    matched = sum(1 for k in patches if not (isinstance(k, str) and k.endswith(".config")))
    layers = {k[: -len(s)] for k in converted for s in _LORA_SUFFIXES if k.endswith(s)}
    total = len(layers)
    if matched == 0:
        raise ValueError(
            f"LoRA '{name}' doesn't match the {model_label} model (0 of {total or '?'} layers matched). "
            f"It was probably made for a different model — remove it or pick a {model_label} LoRA.")
    warning = None
    if total and matched < 0.5 * total:
        warning = (f"LoRA '{name}': only {matched} of {total} layers matched the {model_label} model, "
                   f"so it may have little effect (probably made for a different model version).")
    print(f"  LoRA {name} @ {strength_model}: {matched}/{total or '?'} layers matched")
    new_model, new_clip = comfy.sd.load_lora_for_models(model, clip, lora, strength_model, strength_clip,
                                                        lora_metadata=metadata)
    return new_model, new_clip, warning


# ---------------- generation (background job, polled by the website) ----------------
class LoraItem(BaseModel):
    name: str
    strength: float = 1.0


class GenerateRequest(BaseModel):
    checkpoint: Optional[str] = None
    sampler: str = "dpmpp_2m"
    scheduler: str = "karras"
    steps: int = 25
    cfg: float = 4.5
    width: int = 832
    height: int = 1216
    batch_size: int = 1
    seed: int = -1
    loras: List[LoraItem] = []
    positive: str = ""
    negative: str = ""
    face_fix: bool = False          # redraw faces at full resolution after generating (like ADetailer)
    face_fix_strength: float = 0.4  # how much the face is redrawn: 0.3 subtle ... 0.6 strong


gpu_lock = globals().get("gpu_lock") if isinstance(globals().get("gpu_lock"), type(threading.Lock())) else threading.Lock()
current_job = globals().get("current_job") if isinstance(globals().get("current_job"), dict) else {"id": None}


def progress_hook(value, total, *args, **kwargs):
    job = JOBS.get(current_job["id"])
    if job and total:
        job["step"], job["total_steps"] = int(value), int(total)


comfy.utils.set_progress_bar_global_hook(progress_hook)


def to_data_url(img_tensor):
    arr = (img_tensor.cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
    buf = BytesIO()
    Image.fromarray(arr).save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


# ---------------- Face fix (like ADetailer) ----------------
# After an image is made: find each face, cut out the area around it, redraw it at ~1024 px with the
# same model, LoRAs and prompt (only partly, controlled by the strength), and blend it back in.
# Small faces in full-body images get far more detail and look much more like the LoRA's person.
FACE_MODEL_PATH = "/content/ComfyUI/models/ultralytics/face_yolov8m.pt"
VAEEncode = NODE_CLASS_MAPPINGS["VAEEncode"]()
_face_detector = {"model": None}


def face_fix_available():
    if not os.path.exists(FACE_MODEL_PATH):
        return False
    try:
        import ultralytics  # noqa: F401
        return True
    except Exception:
        return False


def detect_faces(pil_img, conf=0.35, max_faces=4):
    """Face boxes (x1, y1, x2, y2), biggest first. Runs on the CPU so it doesn't take GPU memory."""
    if _face_detector["model"] is None:
        from ultralytics import YOLO
        _face_detector["model"] = YOLO(FACE_MODEL_PATH)
    results = _face_detector["model"].predict(pil_img, conf=conf, verbose=False, device="cpu")
    if not results or results[0].boxes is None:
        return []
    boxes = results[0].boxes.xyxy.cpu().numpy().tolist()
    boxes.sort(key=lambda b: (b[2] - b[0]) * (b[3] - b[1]), reverse=True)
    return boxes[:max_faces]


def tensor_to_pil(img_tensor):
    return Image.fromarray((img_tensor.cpu().numpy() * 255).clip(0, 255).astype(np.uint8))


def fix_faces(pil_img, model, positive, negative, vae_m, seed, req, boxes=None):
    """Returns (image, number of faces redrawn)."""
    from PIL import ImageDraw, ImageFilter
    boxes = detect_faces(pil_img) if boxes is None else boxes
    out = pil_img.copy()
    W, H = out.size
    strength = max(0.1, min(float(req.face_fix_strength or 0.4), 0.8))
    fixed = 0
    for x1, y1, x2, y2 in boxes:
        bw, bh = x2 - x1, y2 - y1
        if bw < 12 or bh < 12:
            continue
        # square area around the face with some hair/neck for context, kept inside the image
        side = int(min(max(bw, bh) * 1.8, W, H))
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        left = int(max(0, min(cx - side / 2, W - side)))
        top = int(max(0, min(cy - side / 2, H - side)))
        crop = out.crop((left, top, left + side, top + side))
        work_size = 1024 if side < 1024 else (side // 8) * 8   # SDXL's native size: where the extra detail comes from
        work = crop.resize((work_size, work_size), Image.Resampling.LANCZOS)
        pixels = torch.from_numpy(np.array(work).astype(np.float32) / 255.0)[None, ]
        latent = VAEEncode.encode(vae_m, pixels)[0]
        samples = KSampler.sample(model, seed, req.steps, req.cfg, req.sampler, req.scheduler,
                                  positive, negative, latent, strength)[0]
        redone = tensor_to_pil(VAEDecode.decode(vae_m, samples)[0][0]).resize((side, side), Image.Resampling.LANCZOS)
        # soft-edged mask so the redrawn area blends in without a visible square
        feather = max(4, side // 10)
        mask = Image.new("L", (side, side), 0)
        ImageDraw.Draw(mask).rectangle((feather, feather, side - feather, side - feather), fill=255)
        mask = mask.filter(ImageFilter.GaussianBlur(feather / 2))
        out.paste(redone, (left, top), mask)
        fixed += 1
    return out, fixed


def pil_to_data_url(pil_img):
    buf = BytesIO()
    pil_img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def run_generate(job_id, req):
    job = JOBS[job_id]
    with gpu_lock:
        current_job["id"] = job_id
        job["status"] = "running"
        try:
            model, clip_m, vae_m = get_checkpoint(req.checkpoint)
            available_loras = set(list_files(LORA_DIR))
            batch = max(1, min(int(req.batch_size or 1), 8))
            w = max(256, min(int(req.width), 2048)) // 8 * 8
            h = max(256, min(int(req.height), 2048)) // 8 * 8
            seed = req.seed if req.seed and req.seed > 0 else random.randint(0, 2**63 - 1)
            if req.face_fix and not face_fix_available():
                raise ValueError("Face fix isn't installed in this notebook — open it fresh from GitHub and run cell 1 again")
            print(f"[{job_id}] {loaded['name']}, {len(req.loras)} LoRA(s), {req.steps} steps, {w}x{h}, batch={batch}"
                  + (f", face fix {req.face_fix_strength}" if req.face_fix else ""))
            warnings = []
            with torch.inference_mode():
                for l in req.loras:
                    if not l.name or l.strength == 0:
                        continue
                    if l.name not in available_loras:
                        raise ValueError(f"LoRA not found on the server: {l.name}")
                    model, clip_m, warning = apply_lora_checked(model, clip_m, l.name, float(l.strength),
                                                                float(l.strength), "SDXL")
                    if warning:
                        warnings.append(warning)
            job["warnings"] = warnings
            with torch.inference_mode():
                positive = CLIPTextEncode.encode(clip_m, req.positive)[0]
                negative = CLIPTextEncode.encode(clip_m, req.negative)[0]
                images, faces_fixed = [], 0
                # one image at a time: lower VRAM use and progress after each image
                for i in range(batch):
                    latent = EmptyLatentImage.generate(w, h, 1)[0]
                    samples = KSampler.sample(model, seed + i, req.steps, req.cfg, req.sampler, req.scheduler,
                                              positive, negative, latent)[0]
                    image = VAEDecode.decode(vae_m, samples)[0][0]
                    if req.face_fix:
                        job["phase"] = "fixing faces"
                        pil, n = fix_faces(tensor_to_pil(image), model, positive, negative, vae_m, seed + i, req)
                        faces_fixed += n
                        print(f"[{job_id}] image {i + 1}: face fix redrew {n} face(s)")
                        images.append(pil_to_data_url(pil))
                    else:
                        images.append(to_data_url(image))
                    job["done"] = i + 1
            job.update(status="done", images=images, seed=seed, width=w, height=h,
                       faces_fixed=faces_fixed if req.face_fix else None)
        except Exception as e:
            traceback.print_exc()
            job.update(status="error", error=str(e))
        finally:
            current_job["id"] = None
            gc.collect()
            torch.cuda.empty_cache()


@app.post("/generate")
def generate(req: GenerateRequest):
    batch = max(1, min(int(req.batch_size or 1), 8))
    job_id = new_job("generate", batch_size=batch, done=0, step=0, total_steps=0, images=[])
    threading.Thread(target=run_generate, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id, "status": "queued", "batch_size": batch}


@app.get("/job/{job_id}")
def get_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        return {"status": "error", "error": last_crash_message() or "Unknown job (the server restarted)"}
    out = {k: v for k, v in job.items() if k != "images"}
    out["job_id"] = job_id
    if job["kind"] == "generate" and job["status"] == "done":
        out["images"] = job["images"]
        out["image"] = job["images"][0] if job["images"] else None
    return out


@app.get("/")
def root():
    return {"status": "ok", "checkpoint": loaded["name"]}


if __name__ == "__main__":
    import uvicorn
    print("Server listening on port 8000.", flush=True)
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

In [ ]:
# Cell 3: start the generation server as a separate program, plus the Cloudflare tunnel.
# If the server crashes (e.g. runs out of memory) only that program dies: it's restarted automatically
# with the same URL, and this notebook keeps running. Re-running this cell restarts the server with the
# current code from cell 2. To stop everything: Runtime → Restart session.
import os, re, sys, time, subprocess, threading, urllib.request
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart", "requests"], check=True)
SERVER_SCRIPT = "/content/biglust_server.py"
SERVER_LOG = "/content/biglust_server.log"
CRASH_FILE = "/content/server_crash.txt"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"


def server_up():
    try:
        return urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=3).status == 200
    except Exception:
        return False


def log_tail(n=15):
    try:
        return "".join(open(SERVER_LOG, errors="replace").readlines()[-n:])
    except OSError:
        return ""


class Supervisor:
    """Runs the server program and restarts it when it crashes (gives up after 4 crashes in 10 minutes)."""

    def __init__(self):
        self.proc, self.thread, self.stopping, self.gave_up, self.crashes = None, None, False, False, []

    def _loop(self):
        while not self.stopping:
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n===== {time.strftime('%H:%M:%S')} starting the server =====\n")
                log.flush()
                self.proc = subprocess.Popen([sys.executable, "-u", SERVER_SCRIPT], cwd="/content/ComfyUI",
                                             stdout=log, stderr=subprocess.STDOUT,
                                             env={**os.environ, "PYTHONUNBUFFERED": "1"})
                code = self.proc.wait()
            if self.stopping:
                break
            why = ("ran out of memory and was stopped by the system" if code in (-9, 137)
                   else f"crashed (exit code {code})")
            self.crashes = [t for t in self.crashes if time.time() - t < 600] + [time.time()]
            if len(self.crashes) >= 4:
                msg = (f"The generation server {why} 4 times in 10 minutes, so it was not restarted. "
                       f"Check the log (cell 4), then run cell 3 again.")
                self.gave_up = True
            else:
                msg = (f"The generation server {why} at {time.strftime('%H:%M')}, so that job was lost. "
                       f"It restarts by itself (models reload in 1-3 minutes). Next time try a smaller size, "
                       f"fewer pictures or batch 1.")
            with open(CRASH_FILE, "w") as fh:
                fh.write(msg)
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n!!!!! {msg}\n")
            if self.gave_up:
                break
            time.sleep(3)

    def start(self):
        self.stopping, self.gave_up = False, False
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def stop(self):
        self.stopping = True
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(30)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        if self.thread:
            self.thread.join(35)


# (Re)start the server program
if "SUPERVISOR" in globals():
    print("Restarting the server with the current code...")
    try:
        SUPERVISOR.stop()
    except Exception:
        pass
if os.path.exists(CRASH_FILE):
    os.remove(CRASH_FILE)
SUPERVISOR = Supervisor()
SUPERVISOR.start()

print("Loading the models in the server program (1-3 minutes)...")
t0 = time.time()
while not server_up():
    if SUPERVISOR.gave_up or time.time() - t0 > 900:
        print("The server did not start. Last lines of its log:\n" + log_tail(30))
        raise RuntimeError("Server failed to start — see the log above")
    time.sleep(5)
print(f"Server is up after {time.time() - t0:.0f}s.")

# Cloudflare tunnel (started once; the URL stays the same when the server restarts)
if not globals().get("TUNNEL_URL"):
    subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
    if not os.path.exists(CLOUDFLARED):
        subprocess.run(["wget", "-q", "-O", CLOUDFLARED,
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod(CLOUDFLARED, 0o755)
    TUNNEL_PROC = subprocess.Popen([CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    t0 = time.time()
    while time.time() - t0 < 60 and not globals().get("TUNNEL_URL"):
        line = TUNNEL_PROC.stdout.readline()
        if not line:
            break
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if m:
            TUNNEL_URL = m.group(0)
    threading.Thread(target=lambda: [None for _ in TUNNEL_PROC.stdout], daemon=True).start()  # keep the pipe drained

if globals().get("TUNNEL_URL"):
    print("=" * 60)
    print(f"READY: {TUNNEL_URL}")
    print("Paste this into the website's Backend URL field.")
    print("This cell is done; the server keeps running in the background.")
    print("Cell 4 shows its log and memory use. To stop it: Runtime → Restart session.")
    print("=" * 60)
else:
    print("Could not start the Cloudflare tunnel. Run this cell again.")

In [ ]:
# Cell 4 (optional): the server's recent log and memory use. Run it again to refresh.
import subprocess
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"System RAM: {(vm.total - vm.available) / 1e9:.1f} / {vm.total / 1e9:.1f} GB")
except Exception:
    pass
gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
if gpu:
    print(f"GPU memory: {gpu}")
sup = globals().get("SUPERVISOR")
if sup is not None:
    running = sup.proc is not None and sup.proc.poll() is None
    print("Server program:", "running" if running else ("stopped — run cell 3" if sup.gave_up else "starting / restarting"))
    print(f"Crashes in the last 10 minutes: {len([t for t in sup.crashes if __import__('time').time() - t < 600])}")
print("-" * 60)
try:
    print("".join(open("/content/biglust_server.log", errors="replace").readlines()[-40:]))
except OSError:
    print("No log yet — run cell 3 first.")